# Семинар Week 2 — линейная регрессия

Сегодня пройдём один маршрут:

**NumPy → формула МНК → собственный estimator → проверка → точная линейная зависимость → `lstsq` → `sklearn` → реальные данные → остатки → новый признак → test.**

Главная цель — увидеть, как формула превращается в работающую модель, а затем использовать ошибки модели для следующего осмысленного эксперимента.

## A. Коротко вспоминаем NumPy

Внутри собственного `fit` не должно неожиданно появиться незнакомое API. Поэтому сначала восстановим только те операции, которые понадобятся в формуле МНК.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

`np.array` — многомерный числовой массив. Для матрицы признаков будем использовать форму `(число объектов, число признаков)`, а для вектора ответов — одномерный массив `(число объектов,)`.

In [ ]:
X_demo = np.array([
    [1.0, 10.0],
    [2.0, 20.0],
    [3.0, 30.0],
])
y_demo = np.array([5.0, 7.0, 9.0])

In [ ]:
X_demo.shape, y_demo.shape

In [ ]:
X_demo[0]
X_demo[:, 0]

In [ ]:
X_demo.T

Если

$$
X\in\mathbb R^{n\times p},
$$

то

$$
X^\top\in\mathbb R^{p\times n},
\qquad
X^\top X\in\mathbb R^{p\times p}.
$$

Перед запуском следующей ячейки назовите размер результата.

In [ ]:
X_demo.T @ X_demo

In [ ]:
ones_demo = np.ones(X_demo.shape[0])
ones_demo

In [ ]:
X_aug_demo = np.column_stack([ones_demo, X_demo])
X_aug_demo

In [ ]:
np.asarray([[1, 2], [3, 4]])

`np.asarray(...)` удобно использовать внутри estimator: вход может быть списком, `DataFrame` или уже массивом NumPy, а матричные операции дальше выполняются в одном формате.

Для квадратной обратимой матрицы $A$ выполняется

$$
A^{-1}A=I.
$$

In [ ]:
A = np.array([
    [2.0, 1.0],
    [1.0, 3.0],
])
A_inv = np.linalg.inv(A)
A_inv

In [ ]:
A_inv @ A

Теперь формула, которую реализуем буквально.

Добавим свободный член как столбец единиц:

$$
X_{\text{aug}}=[\mathbf 1, X].
$$

Если $X_{\text{aug}}$ имеет полный столбцовый ранг, матрица $X_{\text{aug}}^\top X_{\text{aug}}$ обратима и коэффициенты определяются однозначно:

$$
\hat\beta=
(X_{\text{aug}}^\top X_{\text{aug}})^{-1}
X_{\text{aug}}^\top y.
$$

Сейчас это **учебная реализация формулы**, а не рекомендация для практического численного кода.

## B. Пишем `ManualLinearRegression`

Хотим получить знакомый интерфейс:

```python
model = ManualLinearRegression()
model.fit(X, y)
predictions = model.predict(X_new)
```

Во время совместной реализации следим не за синтаксисом класса, а за математическими объектами: какой формы матрицы, где появляются коэффициенты и что остаётся в состоянии модели после `fit`.

In [ ]:
class ManualLinearRegression:
    def __init__(self):
        self.intercept_ = None
        self.coef_ = None

    def fit(self, X, y):
        X = np.asarray(X)
        y = np.asarray(y)

        X_aug = np.column_stack([
            np.ones(X.shape[0]),
            X,
        ])

        gram = X_aug.T @ X_aug
        rhs = X_aug.T @ y
        beta = np.linalg.inv(gram) @ rhs

        self.intercept_ = beta[0]
        self.coef_ = beta[1:]
        return self

    def predict(self, X):
        X = np.asarray(X)
        return self.intercept_ + X @ self.coef_

## C. Проверяем себя на прозрачном примере

Сгенерируем 12 точек примерно по правилу

$$
y=1+2x+\varepsilon.
$$

Шум небольшой, поэтому ожидаем свободный член около 1 и наклон около 2 — но не требуем идеального совпадения.

In [ ]:
rng = np.random.default_rng(2027)

In [ ]:
x_sanity = np.linspace(0, 5, 12)
X_sanity = x_sanity.reshape(-1, 1)

In [ ]:
noise = rng.normal(0, 0.35, size=len(x_sanity))

In [ ]:
y_sanity = 1 + 2 * x_sanity + noise

In [ ]:
plt.scatter(x_sanity, y_sanity)
plt.xlabel("x")
plt.ylabel("y")
plt.title("Прозрачный пример с небольшим шумом")
plt.show()

In [ ]:
manual_model = ManualLinearRegression()

In [ ]:
vars(manual_model)

In [ ]:
manual_model.fit(X_sanity, y_sanity)

In [ ]:
vars(manual_model)

In [ ]:
manual_model.intercept_

In [ ]:
manual_model.coef_

In [ ]:
manual_pred = manual_model.predict(X_sanity)
manual_pred

Теперь сравним нашу реализацию с `sklearn.linear_model.LinearRegression`.

Документация: [https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html)

На этом этапе полезны знакомые `fit`, `predict` и обученные атрибуты `coef_`, `intercept_`; для плотной матрицы признаков estimator также хранит информацию о ранге и сингулярных значениях.

In [ ]:
sklearn_model = LinearRegression()

In [ ]:
sklearn_model.fit(X_sanity, y_sanity)

In [ ]:
sklearn_model.intercept_, sklearn_model.coef_

In [ ]:
sklearn_pred = sklearn_model.predict(X_sanity)
sklearn_pred

In [ ]:
pd.DataFrame({
    "manual": manual_pred,
    "sklearn": sklearn_pred,
})

In [ ]:
np.allclose(manual_pred, sklearn_pred)

## D. Ломаем inverse-реализацию

Добавим второй признак, который не содержит новой информации:

$$
x_2=2x_1.
$$

Перед запуском подумайте:

- стало ли больше информации о целевой переменной;
- что произошло с числом столбцов;
- могут ли разные пары коэффициентов давать один и тот же прогноз?

In [ ]:
X_collinear = np.column_stack([
    x_sanity,
    2 * x_sanity,
])

In [ ]:
X_collinear

In [ ]:
X_collinear.shape

In [ ]:
broken_model = ManualLinearRegression()
broken_model.fit(X_collinear, y_sanity)

Падение говорит не о том, что предиктивная задача внезапно исчезла. Мы нарушили условие, при котором пытались получить **единственный** вектор коэффициентов через явное обращение матрицы.

In [ ]:
np.linalg.matrix_rank(X_collinear)

In [ ]:
X_collinear.shape[1]

Два разумных программных поведения:

1. проверить условие и вернуть понятную ошибку;
2. решать исходную задачу наименьших квадратов численным методом, не вычисляя явную обратную матрицу.

Пойдём по второму пути.

## E. `np.linalg.lstsq`: решаем least-squares задачу напрямую

Мы по-прежнему хотим решить

$$
\hat\beta\in\arg\min_\beta\|y-X_{\text{aug}}\beta\|_2^2.
$$

`np.linalg.lstsq` принимает саму матрицу и вектор ответов и возвращает решение задачи наименьших квадратов.

Документация: [https://numpy.org/doc/stable/reference/generated/numpy.linalg.lstsq.html](https://numpy.org/doc/stable/reference/generated/numpy.linalg.lstsq.html)

Контракт:

```python
beta, residuals, rank, singular_values = np.linalg.lstsq(...)
```

In [ ]:
X_aug_collinear = np.column_stack([
    np.ones(X_collinear.shape[0]),
    X_collinear,
])

In [ ]:
beta, residuals, rank, singular_values = np.linalg.lstsq(
    X_aug_collinear,
    y_sanity,
    rcond=None,
)

In [ ]:
beta

In [ ]:
residuals

In [ ]:
rank

In [ ]:
singular_values

In [ ]:
lstsq_pred = X_aug_collinear @ beta
lstsq_pred

При точной линейной зависимости коэффициенты не обязаны быть единственными. Численный метод возвращает одно подходящее решение, а нас в этой части курса прежде всего интересует получившееся правило прогноза.

### Та же модель — другой способ вычисления коэффициентов

Внешний интерфейс не меняется. В готовой второй версии класса мы заменяем только внутреннюю реализацию `fit`.

In [ ]:
class ManualLinearRegression:
    def __init__(self):
        self.intercept_ = None
        self.coef_ = None
        self.rank_ = None
        self.singular_ = None

    def fit(self, X, y):
        X = np.asarray(X)
        y = np.asarray(y)

        X_aug = np.column_stack([
            np.ones(X.shape[0]),
            X,
        ])

        beta, _, rank, singular_values = np.linalg.lstsq(
            X_aug,
            y,
            rcond=None,
        )

        self.intercept_ = beta[0]
        self.coef_ = beta[1:]
        self.rank_ = rank
        self.singular_ = singular_values
        return self

    def predict(self, X):
        X = np.asarray(X)
        return self.intercept_ + X @ self.coef_

In [ ]:
manual_lstsq_model = ManualLinearRegression()
manual_lstsq_model.fit(X_collinear, y_sanity)

In [ ]:
manual_collinear_pred = manual_lstsq_model.predict(X_collinear)
manual_collinear_pred

In [ ]:
sklearn_collinear_model = LinearRegression()
sklearn_collinear_model.fit(X_collinear, y_sanity)

In [ ]:
sklearn_collinear_pred = sklearn_collinear_model.predict(X_collinear)
sklearn_collinear_pred

In [ ]:
pd.DataFrame({
    "manual_lstsq": manual_collinear_pred,
    "sklearn": sklearn_collinear_pred,
})

In [ ]:
np.allclose(manual_collinear_pred, sklearn_collinear_pred)

In [ ]:
order = np.argsort(X_collinear[:, 0])

plt.scatter(X_collinear[:, 0], y_sanity, label="Данные")
plt.plot(
    X_collinear[order, 0],
    manual_collinear_pred[order],
    label="ManualLinearRegression",
)
plt.plot(
    X_collinear[order, 0],
    sklearn_collinear_pred[order],
    linestyle="--",
    label="sklearn",
)
plt.xlabel("x₁")
plt.ylabel("y")
plt.title("Прогнозы при точной линейной зависимости признаков")
plt.legend()
plt.show()

Мы смогли получить прогнозы даже при точной зависимости признаков. Но это **не** означает, что в реальной ML-системе стоит бездумно оставлять детерминированно дублирующие признаки.

Практически:

- явное обращение матрицы обычно не используют как практический численный способ решения;
- точно избыточный признак чаще разумно не генерировать или удалить;
- сильные связи могут делать отдельные коэффициенты нестабильными и осложнять интерпретацию;
- позже мы увидим регуляризацию как ещё один инструмент работы с линейными моделями;
- особенно важно, будет ли связь между признаками сохраняться на будущих данных.

Для прогнозирования ключевой вывод сейчас такой: **семейство моделей, задача наименьших квадратов, численный метод и интерфейс `fit/predict` — разные уровни описания одной процедуры.**

## F. Реальные данные: Auto MPG

Теперь проведём компактный сквозной эксперимент по регрессии.

**Задача:** по характеристикам автомобиля предсказать `mpg` — сколько миль автомобиль проезжает на одном галлоне топлива.

Источник: [UCI Auto MPG](https://archive.ics.uci.edu/dataset/9/auto+mpg). В исходной таблице 398 автомобилей.

Столбцы:

- `cylinders` — число цилиндров двигателя;
- `displacement` — рабочий объём двигателя;
- `horsepower` — мощность двигателя;
- `weight` — масса автомобиля;
- `acceleration` — характеристика ускорения;
- `model_year` — модельный год;
- `origin` — код региона происхождения;
- `car_name` — название автомобиля;
- `mpg` — **целевая переменная**, топливная экономичность в милях на галлон.

Файл рядом с notebook уже технически перепакован из legacy-формата в обычный CSV, но содержательная очистка ещё не сделана.

In [ ]:
data = pd.read_csv("data/auto_mpg.csv")

In [ ]:
data.shape

In [ ]:
data.head()

In [ ]:
data.dtypes

In [ ]:
data.isna().sum()

В `horsepower` всего несколько пропусков. Заполнение пропусков мы ещё не изучали, поэтому для компактного примера просто удалим эти строки. Это локальное учебное решение, а не универсальная рекомендация.

In [ ]:
data_clean = data.dropna(subset=["horsepower"]).copy()

In [ ]:
data_clean.shape

`car_name` — текстовое название/идентификатор автомобиля. `origin` кодирует категорию происхождения. На Week 2 мы ещё не строим представление текстовых и категориальных признаков, поэтому первый эксперимент проведём только на числовых признаках ниже.

`cylinders` и `model_year` тоже дискретны, но сейчас сознательно используем их как числовые признаки первой простой модели.

In [ ]:
feature_columns = [
    "cylinders",
    "displacement",
    "horsepower",
    "weight",
    "acceleration",
    "model_year",
]

In [ ]:
data_clean[feature_columns + ["mpg"]].describe()

In [ ]:
data_clean["mpg"].hist(bins=20)
plt.xlabel("mpg")
plt.ylabel("Число автомобилей")
plt.title("Распределение целевой переменной")
plt.show()

Гистограмма нужна здесь только чтобы почувствовать диапазон целевой переменной и увидеть, с какими числовыми значениями мы работаем. Она сама по себе не выбирает модель и не подсказывает будущие преобразования признаков.

## G. Обучающая и тестовая части

Сначала формируем матрицу признаков и целевую переменную, затем один раз фиксируем `train_test_split`.

До финального блока тестовую часть **не используем для выбора новых признаков или преобразований**.

In [ ]:
X = data_clean[feature_columns]
y = data_clean["mpg"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=2027,
)

In [ ]:
X_train.shape, X_test.shape

### Бейзлайн

Начнём с простого постоянного прогноза: среднее значение `mpg` на обучающей выборке.

In [ ]:
baseline_model = DummyRegressor(strategy="mean")

In [ ]:
baseline_model.fit(X_train, y_train)

In [ ]:
baseline_train_pred = baseline_model.predict(X_train)

In [ ]:
baseline_train_mae = mean_absolute_error(y_train, baseline_train_pred)
baseline_train_mae

In [ ]:
baseline_train_rmse = root_mean_squared_error(y_train, baseline_train_pred)
baseline_train_rmse

In [ ]:
baseline_train_r2 = r2_score(y_train, baseline_train_pred)
baseline_train_r2

Мы смотрим на несколько знакомых сводок ошибки:

- **MAE** — средний абсолютный размер промаха;
- **RMSE** сильнее реагирует на крупные промахи;
- **$R^2$** — знакомая относительная характеристика качества.

Прикладная стоимость ошибок здесь не задана, поэтому не объявляем одну из этих метрик единственно правильной. На следующей неделе отдельно разберём, что означает выбор функции потерь.

## H. Первая линейная модель

In [ ]:
linear_model = LinearRegression()

In [ ]:
linear_model.fit(X_train, y_train)

In [ ]:
linear_train_pred = linear_model.predict(X_train)

In [ ]:
linear_train_mae = mean_absolute_error(y_train, linear_train_pred)
linear_train_mae

In [ ]:
linear_train_rmse = root_mean_squared_error(y_train, linear_train_pred)
linear_train_rmse

In [ ]:
linear_train_r2 = r2_score(y_train, linear_train_pred)
linear_train_r2

In [ ]:
pd.DataFrame({
    "модель": ["Средний прогноз", "Линейная регрессия"],
    "MAE train": [baseline_train_mae, linear_train_mae],
    "RMSE train": [baseline_train_rmse, linear_train_rmse],
    "R² train": [baseline_train_r2, linear_train_r2],
})

## I. Что скрывает одно число качества?

Для обучающего объекта остаток равен

$$
r_i=y_i-\hat y_i.
$$

Теперь вопрос меняется: **есть ли в ошибках линейной модели систематическая структура?**

In [ ]:
train_residuals = y_train - linear_train_pred

In [ ]:
plt.scatter(linear_train_pred, train_residuals, alpha=0.65)
plt.axhline(0, linestyle="--")
plt.xlabel("Прогноз")
plt.ylabel("Остаток")
plt.title("Остатки против прогнозов · train")
plt.show()

In [ ]:
def plot_residuals_by_feature(X, residuals, columns):
    n_cols = 2
    n_rows = int(np.ceil(len(columns) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(11, 3.2 * n_rows))
    axes = np.asarray(axes).reshape(-1)

    for ax, column in zip(axes, columns):
        ax.scatter(X[column], residuals, alpha=0.6, s=24)
        ax.axhline(0, linestyle="--")
        ax.set_xlabel(column)
        ax.set_ylabel("Остаток")

    for ax in axes[len(columns):]:
        ax.axis("off")

    fig.suptitle("Остатки по отдельным признакам · train", y=1.01)
    fig.tight_layout()
    plt.show()

In [ ]:
plot_residuals_by_feature(
    X_train,
    train_residuals,
    feature_columns,
)

### Небольшое исследование

Работайте **только с train**.

1. Выберите один признак, относительно которого остатки показывают заметную нелинейную структуру.
2. Опишите рисунок словами.
3. Если видите кривизну, проверьте простую гипотезу: добавить квадрат этого признака.
4. Сравните train-метрики и новый график остатков.

Не обязательно выбирать тот же признак, что соседняя группа.

In [ ]:
feature = "..."  # впишите выбранный признак

In [ ]:
def add_squared_feature(X, feature):
    X_new = X.copy()
    X_new[f"{feature}_sq"] = X_new[feature] ** 2
    return X_new

Функция нужна не ради архитектуры: выбранное правило позже должно **точно так же** примениться к test. Это ещё не `sklearn.Pipeline`: функция ничего не обучает и не хранит состояние.

In [ ]:
X_train_squared = add_squared_feature(X_train, feature)

In [ ]:
squared_model = LinearRegression()

In [ ]:
squared_model.fit(X_train_squared, y_train)

In [ ]:
squared_train_pred = squared_model.predict(X_train_squared)

In [ ]:
squared_train_mae = mean_absolute_error(y_train, squared_train_pred)
squared_train_mae

In [ ]:
squared_train_rmse = root_mean_squared_error(y_train, squared_train_pred)
squared_train_rmse

In [ ]:
squared_train_r2 = r2_score(y_train, squared_train_pred)
squared_train_r2

In [ ]:
pd.DataFrame({
    "модель": ["Линейная регрессия", f"Линейная + {feature}²"],
    "MAE train": [linear_train_mae, squared_train_mae],
    "RMSE train": [linear_train_rmse, squared_train_rmse],
    "R² train": [linear_train_r2, squared_train_r2],
})

In [ ]:
squared_train_residuals = y_train - squared_train_pred

plt.scatter(X_train[feature], squared_train_residuals, alpha=0.65)
plt.axhline(0, linestyle="--")
plt.xlabel(feature)
plt.ylabel("Остаток")
plt.title(f"Остатки после добавления {feature}² · train")
plt.show()

К этому моменту у нас сохранены разные объекты моделей:

- `baseline_model`;
- `linear_model`;
- `squared_model`.

Не переиспользуем одно имя `model`: в конце хотим вернуться ко всем трём вариантам и сравнить их.

## J. Финальное сравнение на test

Все модели и преобразования выше были сформулированы **без просмотра test**. Теперь нормально один раз оценить их на тестовой части и сравнить.

После просмотра результатов не начинаем придумывать новые квадраты по тому, что случайно оказалось лучше именно на этом test, если хотим сохранить его как независимую финальную проверку.

In [ ]:
baseline_test_pred = baseline_model.predict(X_test)

In [ ]:
linear_test_pred = linear_model.predict(X_test)

In [ ]:
X_test_squared = add_squared_feature(X_test, feature)

In [ ]:
squared_test_pred = squared_model.predict(X_test_squared)

In [ ]:
baseline_test_metrics = {
    "модель": "Средний прогноз",
    "MAE test": mean_absolute_error(y_test, baseline_test_pred),
    "RMSE test": root_mean_squared_error(y_test, baseline_test_pred),
    "R² test": r2_score(y_test, baseline_test_pred),
}

In [ ]:
linear_test_metrics = {
    "модель": "Линейная регрессия",
    "MAE test": mean_absolute_error(y_test, linear_test_pred),
    "RMSE test": root_mean_squared_error(y_test, linear_test_pred),
    "R² test": r2_score(y_test, linear_test_pred),
}

In [ ]:
squared_test_metrics = {
    "модель": f"Линейная + {feature}²",
    "MAE test": mean_absolute_error(y_test, squared_test_pred),
    "RMSE test": root_mean_squared_error(y_test, squared_test_pred),
    "R² test": r2_score(y_test, squared_test_pred),
}

In [ ]:
test_results = pd.DataFrame([
    baseline_test_metrics,
    linear_test_metrics,
    squared_test_metrics,
])

test_results

### Вопросы для самопроверки

- Линейная модель лучше бейзлайна на новых объектах?
- Какую систематическую структуру мы увидели в train-остатках?
- Улучшение после добавления квадрата сохранилось на test?
- Какие варианты получили другие группы?
- Что будет не так, если теперь посмотреть на test-таблицу, придумать ещё пять преобразований и продолжать называть этот же test независимой финальной проверкой?

## K. Один знакомый взгляд из `statsmodels`

В `sklearn` мы работали с линейной моделью прежде всего как с предиктором. В эконометрике вы могли видеть другой интерфейс — `statsmodels`, где после оценки модели доступна подробная статистическая сводка.

Сейчас ничего из неё системно не разбираем: просто получим **ту же базовую линейную модель на исходных признаках** и посмотрим, как выглядит `summary()`.

In [ ]:
import statsmodels.api as sm

In [ ]:
X_sm = sm.add_constant(X_train)

In [ ]:
statsmodels_model = sm.OLS(y_train, X_sm).fit()

In [ ]:
statsmodels_model.summary()

## Итог

Мы прошли путь от явной формулы коэффициентов до реального regression experiment:

1. собрали матричную формулу на NumPy;
2. реализовали упрощённый estimator;
3. проверили его относительно `sklearn`;
4. увидели, почему точная линейная зависимость ломает inverse-формулу, но не обязана уничтожать возможность строить прогнозы;
5. перешли к `np.linalg.lstsq`;
6. на реальных данных сравнили бейзлайн и линейную модель;
7. использовали остатки, чтобы предложить новый признак;
8. проверили заранее сформулированные модели на test.